In [ ]:
import os, pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
import manifold_dynamics.tuning_utils as tut

In [ ]:
DATA_DIR = '../../datasets/NNN/'
# IMG_DIR = '../../datasets/NNN/NSD1000_LOC'

CAT = 'face'
dat = pd.read_pickle(os.path.join(DATA_DIR, (f'{CAT}_roi_data.pkl')))
ROI_LIST = list(dat['roi'].unique())
print(f'Unique {CAT} ROIs: {ROI_LIST}')

# optimal k: the optimal manifold scale at which we should conduct analyses
# see ./pause.ipynb
with open(f'../../datasets/NNN/{CAT}_mins.pkl', 'rb') as f:
    mins = pickle.load(f)
SCALES = {k: v[0] for k,v in mins.items()}

In [ ]:
ROI = 'Unknown_19_F'  # Unknown_19_F, MF1_7_F, MF1_8_F, MF1_9_F, AF3_18_F, MB1_3_B

tstart = 100
tend=350
scale = SCALES[ROI]

# find top images
rng = np.random.default_rng(0)
scores = tut.landscape(dat, ROI)
order = np.argsort(scores)[::-1]

X = tut.response_array(dat, ROI)
img_rdm, _ = tut.time_avg_rdm(dat, ROI, window=slice(tstart, tend), images=order[:scale])

In [ ]:
print(f'Top k images: {order[:10]} ...')
print(f'Number of responsive units: {X.shape[0]} ...')

print(f'Total number of units: {len(dat[dat['roi']==ROI])} ...')

In [ ]:
# VIZ top K time traces
for ROI in ROI_LIST:
    scale = SCALES[ROI]
    scores = tut.landscape(dat, ROI)
    order = np.argsort(scores)[::-1]
    X = tut.response_array(dat, ROI)
    
    fig, ax = plt.subplots(1,1)
    customp = sns.color_palette('husl', scale)
    
    for i, iid in enumerate(order[:scale]):
        resp = np.nanmean(X[:, :, iid], axis=0)
        ax.plot(resp, alpha=0.2, c=customp[i])
        
    ax.plot(np.nanmean(X[:, :, order[:scale]], axis=(0,2)), c='black')
    ax.set_title(f'{ROI} top {scale}')
    ax.set_xlabel('Time (msec)')
    plt.show()  

In [ ]:
# some Time x Time plot for single images
from scipy.spatial.distance import pdist, cdist, squareform
from scipy.stats import zscore

img = order[4]
rdv = pdist(X[:, :,img].T, metric='correlation')
rdm = squareform(rdv)
sns.heatmap(rdm)
plt.show()

sns.heatmap(X[:, :, img], square=True, cbar=False)
plt.show()

rdvs = []
for i in range(scale):
    img = order[i]
    rdv = pdist(X[:, :,img].T, metric='correlation')
    rdv = zscore(rdv)
    rdvs.append(rdv)

rdm = squareform(np.mean(np.array(rdvs), axis=0))
sns.heatmap(rdm)
plt.show()

# sns.heatmap(np.mean(X[:, :, img], axis=2), square=True, cbar=False)
# plt.show()

img = order[:1072]
rdv = pdist(np.mean(X[:, :,img], axis=2).T, metric='correlation')
rdm = squareform(rdv)
sns.heatmap(rdm)
plt.show()

sns.heatmap(np.mean(X[:, :, img], axis=2), square=True, cbar=False)
plt.show()